# 2SeleksiKeyword

In [ ]:
"""
=============================================================================
PIPELINE VALIDASI KATA KUNCI & PENYARINGAN KORPUS — LAPIS 1 + 2 + 3
=============================================================================
SATU SEL, SEKALI RUN, semua angka Bab 4 keluar.

Tidak ada daftar kata kunci yang ditulis manual selain 58 KANDIDAT AWAL.
Kata kunci yang lolos Lapis 1 mengalir otomatis ke Lapis 2, dan yang lolos
Lapis 2 mengalir otomatis ke Lapis 3. Berapa pun jumlah yang lolos, seluruh
tabel menyesuaikan sendiri.

Keluaran:
    Tabel 4.1  Distribusi kandidat per kelompok tematik
    Tabel 4.2  Hasil Lapis 1 (TF-IDF)
    Tabel 4.3  Hasil Lapis 2 (Spearman)
    Tabel 4.4  Koreksi False Discovery Rate
    Tabel 4.5  Daftar kata kunci final
    Tabel 4.6  Ringkasan pengumpulan dan penyaringan berita

Efisiensi: satu matriks hitung n-gram dipakai bersama oleh ketiga lapis,
sehingga tidak ada perulangan Python atas 300 ribu dokumen dan tidak ada
pembangunan vocabulary 7,8 juta fitur.
=============================================================================
"""

import re
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.feature_extraction.text import CountVectorizer
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.width', 250)
pd.set_option('display.max_rows', 200)


# =============================================================================
# 0. PARAMETER
# =============================================================================
TICKER          = "ADRO.JK"
MIN_TEXT_LEN    = 10       # dokumen dengan teks bersih <= ini dibuang
MIN_DF          = 3        # Manning et al. (2008) — vocabulary pruning
MAX_DF_PCT      = 95.0     # Salton & Buckley (1988) — quasi-stopword
ANOMALI_PCT     = 20.0     # di atas ini ditandai anomali dan tidak diteruskan
ALPHA           = 0.05     # Fisher (1925)
FDR_LEVELS      = [0.05, 0.10, 0.20]

# Pencocokan kata kunci.
#   'word'      → batas kata (n-gram token). DIREKOMENDASIKAN.
#   'substring' → perilaku lama; 'karbon' ikut tercocok di dalam
#                 'dekarbonisasi', dan 'co2' yang tercuci menjadi 'co'
#                 tercocok di dalam 'detikcom', 'cocok', 'coba', dst.
MATCH_MODE      = 'word'

# Perlakuan artikel yang terbit di luar hari perdagangan.
#   'forward' → dipetakan ke hari perdagangan berikutnya (sesuai Bab 3)
#   'drop'    → dibuang (perilaku lama)
WEEKEND_RULE    = 'forward'

FORCE_RELOAD    = False    # True = unduh ulang meski sudah ada di memori

FILE_IDS = {
    '2016': '1ZkRSFCWHnMXbUgvcP5HiDNhF7sQle5jN',
    '2017': '1q1ta0a6zZd-QnyPvuImcqmvo9X-sjTPY',
    '2018': '1CFMVJ3SleLZVQ3NkT_P9veNL_nVl4KJ7',
    '2019': '1TTV7IGu8O61_H-nIiq7Npl1HsZuMMPRS',
    '2020': '18T_pUFgKLEdRlf5bNh_99wKvWeB2Bmc_',
    '2021': '1ie9F0BjWHM7zgskLetueXkPEd5uYU94j',
    '2022': '18AzOZrjDrwU0gbn_SXOsHcIw7XQUFWCg',
    '2023': '1WtImh6T3wUFnmxl4WmqZuo0kGNyOn-RW',
    '2024': '1RCM7_7n1LByNK8xt60XV05dzbLB1uPMp',
    '2025': '1dXTozJhPKrLnTrGHahTW95_E4nqWrSzZ',
}

CANDIDATE_KEYWORDS = {
    "Energi Terbarukan": [
        "energi terbarukan", "energi baru terbarukan", "ebt",
        "energi surya", "panel surya", "tenaga surya",
        "energi angin", "turbin angin", "panas bumi", "geothermal",
        "hidroelektrik", "biomassa", "bioenergi", "biofuel",
        "transisi energi", "bauran energi", "ebtke",
        "plts", "pltb", "pltp", "plta",
    ],
    "Energi Fosil": [
        "batu bara", "batubara", "coal", "bahan bakar fosil",
        "gas alam", "lng", "lpg",
        "minyak bumi", "minyak mentah", "crude oil",
        "bbm", "harga minyak", "pertamina",
        "pltu", "pembangkit batu bara",
    ],
    "Kelistrikan": [
        "listrik", "pln", "ketenagalistrikan",
        "pembangkit listrik", "tarif listrik",
        "jaringan listrik", "transmisi listrik",
    ],
    "Nuklir": ["nuklir", "tenaga nuklir", "pltn", "reaktor nuklir"],
    "Kebijakan & Lingkungan": [
        "net zero emission", "nze", "karbon", "emisi karbon",
        "co2", "dekarbonisasi", "perubahan iklim",
        "green energy", "energi hijau", "esdm", "kementerian energi",
    ],
}


# =============================================================================
# UTILITAS
# =============================================================================
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def _id(x, dec=0):
    return f"{x:,.{dec}f}".replace(',', '#').replace('.', ',').replace('#', '.')


def head(title):
    print("\n" + "=" * 96)
    print(title)
    print("=" * 96)


def benjamini_hochberg(pvals, q):
    """Kembalikan mask boolean uji yang lolos koreksi FDR pada level q."""
    p = np.asarray(pvals, dtype=float)
    m = p.size
    order = np.argsort(p)
    thresh = q * (np.arange(1, m + 1) / m)
    passed = p[order] <= thresh
    k = np.max(np.nonzero(passed)[0]) + 1 if passed.any() else 0
    mask = np.zeros(m, dtype=bool)
    mask[order[:k]] = True
    return mask


ALL_CANDIDATES, GROUP_MAP = [], {}
for g, kws in CANDIDATE_KEYWORDS.items():
    for kw in kws:
        ALL_CANDIDATES.append(kw)
        GROUP_MAP[kw] = g

head("TABEL 4.1 — DISTRIBUSI KANDIDAT KATA KUNCI PER KELOMPOK TEMATIK")
tabel41 = pd.DataFrame(
    [{'Kelompok Tematik': g, 'Jumlah Kandidat': len(k)} for g, k in CANDIDATE_KEYWORDS.items()]
).sort_values('Jumlah Kandidat', ascending=False).reset_index(drop=True)
tabel41.index = range(1, len(tabel41) + 1)
print(tabel41.to_string())
print(f"{'Total':<28}{len(ALL_CANDIDATES):>6}")


# =============================================================================
# 1. MUAT DATA — sekali saja, sisanya pakai cache di memori
# =============================================================================
head("1. PEMUATAN DATA")

if 'DF_NEWS_RAW' in globals() and not FORCE_RELOAD:
    print(f"Korpus mentah diambil dari memori : {len(DF_NEWS_RAW):,} artikel")
else:
    import gdown
    _frames = []
    for year, fid in FILE_IDS.items():
        out = f"berita_{year}.csv"
        gdown.download(f"https://drive.google.com/uc?id={fid}", out, quiet=True)
        _d = pd.read_csv(out)
        _d['year'] = year
        _frames.append(_d)
        print(f"  {year} : {len(_d):>7,} artikel")
    DF_NEWS_RAW = pd.concat(_frames, ignore_index=True)
    del _frames
    print(f"Total artikel mentah : {len(DF_NEWS_RAW):,}")

N_RAW = len(DF_NEWS_RAW)

DATE_COL = next((c for c in ['tanggal', 'date', 'datetime', 'published', 'waktu']
                 if c in DF_NEWS_RAW.columns), None)
TEXT_COL = next((c for c in ['isi', 'content', 'text', 'body']
                 if c in DF_NEWS_RAW.columns), None)
TITLE_COL = next((c for c in ['judul', 'title', 'headline'] if c in DF_NEWS_RAW.columns), None)
URL_COL = next((c for c in ['url', 'link', 'tautan'] if c in DF_NEWS_RAW.columns), None)
if DATE_COL is None or TEXT_COL is None:
    raise ValueError(f"Kolom tanggal/teks tidak terdeteksi. Kolom: {list(DF_NEWS_RAW.columns)}")
print(f"Kolom  → tanggal: '{DATE_COL}'  teks: '{TEXT_COL}'  "
      f"judul: '{TITLE_COL}'  url: '{URL_COL}'")

# Pembersihan
_txt = DF_NEWS_RAW[TEXT_COL].map(preprocess_text)
_dt = pd.to_datetime(DF_NEWS_RAW[DATE_COL], errors='coerce')
if getattr(_dt.dtype, 'tz', None) is not None:
    _dt = _dt.dt.tz_localize(None)

mask_ok = (_txt.str.len() > MIN_TEXT_LEN) & _dt.notna()
n_teks_kosong = int((_txt.str.len() <= MIN_TEXT_LEN).sum())
n_tgl_invalid = int(_dt.isna().sum())

DOCS = pd.DataFrame({'text': _txt[mask_ok].values,
                     'dt': _dt[mask_ok].dt.normalize().values})
IDX_OK = DF_NEWS_RAW.index[mask_ok]
N_VALID = len(DOCS)
n_dup_teks = int(DOCS['text'].duplicated().sum())
n_dup_url = int(DF_NEWS_RAW.loc[IDX_OK].duplicated(subset=[URL_COL]).sum()) if URL_COL else None

print(f"\nArtikel teks kosong / terlalu pendek : {n_teks_kosong:,}")
print(f"Artikel tanggal tidak terbaca        : {n_tgl_invalid:,}")
print(f"Dokumen valid                        : {N_VALID:,}")
print(f"Duplikat isi (diagnostik, tidak dibuang) : {n_dup_teks:,}")
print(f"Duplikat URL (diagnostik, tidak dibuang) : "
      f"{n_dup_url if n_dup_url is not None else 'kolom URL tidak ada'}")
print(f"Rentang tanggal : {DOCS['dt'].min().date()} → {DOCS['dt'].max().date()}")

# Harga saham
if 'DF_STOCK' in globals() and not FORCE_RELOAD:
    print(f"\nData saham dari memori : {len(DF_STOCK):,} hari perdagangan")
else:
    import yfinance as yf
    _s = yf.Ticker(TICKER).history(start=str(DOCS['dt'].min().date()),
                                   end=str(DOCS['dt'].max().date() + pd.Timedelta(days=1)))
    _s = _s.reset_index()
    DF_STOCK = pd.DataFrame({
        'dt': pd.to_datetime(pd.Series(_s['Date']).dt.tz_localize(None).dt.normalize()),
        'close': _s['Close'].values})
    DF_STOCK['return'] = DF_STOCK['close'].pct_change()
    DF_STOCK['abs_return'] = DF_STOCK['return'].abs()
    DF_STOCK = DF_STOCK.dropna().sort_values('dt').reset_index(drop=True)
    print(f"\nData saham {TICKER} : {len(DF_STOCK):,} hari perdagangan "
          f"({DF_STOCK['dt'].min().date()} → {DF_STOCK['dt'].max().date()})")

N_TRADING = len(DF_STOCK)


# =============================================================================
# 2. MATRIKS HITUNG N-GRAM — dipakai bersama oleh Lapis 1, 2, dan 3
# =============================================================================
head("2. PERHITUNGAN FREKUENSI KANDIDAT KATA KUNCI")

KW_CLEAN = {kw: preprocess_text(kw) for kw in ALL_CANDIDATES}
terdistorsi = [k for k, v in KW_CLEAN.items() if v != k.lower()]
if terdistorsi:
    print("Kandidat yang berubah bentuk setelah pembersihan teks:")
    for k in terdistorsi:
        print(f"   '{k}'  →  '{KW_CLEAN[k]}'")
    print("   Perhatikan kandidat yang menyusut drastis: bentuk bersihnya dapat")
    print("   tercocok di dalam kata lain dan menghasilkan document frequency palsu.")

KW_LIST = list(KW_CLEAN.values())
VOCAB = sorted(set(v for v in KW_LIST if v))

print(f"\nMode pencocokan : '{MATCH_MODE}'")
if MATCH_MODE == 'word':
    cv = CountVectorizer(vocabulary=VOCAB, ngram_range=(1, 3), lowercase=False)
    X = cv.fit_transform(DOCS['text'].values).astype(np.int32)
    col_of = {t: i for i, t in enumerate(cv.get_feature_names_out())}
else:
    cols = []
    for t in VOCAB:
        cols.append(DOCS['text'].str.count(re.escape(t)).values.astype(np.int32))
    from scipy import sparse
    X = sparse.csr_matrix(np.column_stack(cols))
    col_of = {t: i for i, t in enumerate(VOCAB)}

CNT = np.asarray(X.todense())          # dokumen x kandidat
BIN = (CNT > 0)
DF_COUNT = BIN.sum(axis=0)             # document frequency per kandidat
print(f"Matriks hitung : {CNT.shape[0]:,} dokumen x {CNT.shape[1]} kandidat")


# =============================================================================
# 3. LAPIS 1 — TF-IDF
# =============================================================================
head("3. LAPIS 1 — VALIDASI TF-IDF")

# w(t,d) = (1 + ln tf) * idf(t),  idf(t) = ln((1+N)/(1+df)) + 1   [smooth idf]
# Nilai yang dilaporkan adalah rata-rata w(t,d) atas seluruh dokumen.
with np.errstate(divide='ignore'):
    sub_tf = np.where(CNT > 0, 1.0 + np.log(np.maximum(CNT, 1)), 0.0)
idf = np.log((1.0 + N_VALID) / (1.0 + DF_COUNT)) + 1.0
mean_tfidf = (sub_tf * idf).mean(axis=0)

rows = []
for kw in ALL_CANDIDATES:
    j = col_of.get(KW_CLEAN[kw])
    df_c = int(DF_COUNT[j]) if j is not None else 0
    pct = df_c / N_VALID * 100
    score = float(mean_tfidf[j]) if j is not None else 0.0
    tolak = []
    if score <= 0 or df_c == 0:
        tolak.append('tfidf=0')
    if df_c < MIN_DF:
        tolak.append(f'df<{MIN_DF}')
    if pct > MAX_DF_PCT:
        tolak.append(f'df>{MAX_DF_PCT:.0f}%')
    if pct > ANOMALI_PCT:
        tolak.append(f'anomali>{ANOMALI_PCT:.0f}%')
    rows.append({'keyword': kw, 'kelompok': GROUP_MAP[kw], 'keyword_clean': KW_CLEAN[kw],
                 'mean_tfidf': round(score, 6), 'doc_freq': df_c,
                 'doc_freq_pct': round(pct, 2),
                 'status': 'LOLOS' if not tolak else 'TOLAK',
                 'alasan': '; '.join(tolak)})

df_l1 = pd.DataFrame(rows).sort_values(['kelompok', 'mean_tfidf'], ascending=[True, False])
KEYWORDS_L1 = df_l1.loc[df_l1['status'] == 'LOLOS', 'keyword'].tolist()

print(f"\nTABEL 4.2 — Hasil Validasi Lapis 1 (seluruh {len(ALL_CANDIDATES)} kandidat)\n")
print(df_l1[['keyword', 'kelompok', 'mean_tfidf', 'doc_freq_pct', 'status', 'alasan']]
      .to_string(index=False))
print(f"\nLolos Lapis 1 : {len(KEYWORDS_L1)} kata kunci")
_tolak = df_l1[df_l1['status'] == 'TOLAK']
if len(_tolak):
    print("Ditolak       :")
    for _, r in _tolak.iterrows():
        print(f"   {r['keyword']:<24} ({r['alasan']}; doc_freq {r['doc_freq_pct']}%)")


# =============================================================================
# 4. LAPIS 2 — SPEARMAN RANK CORRELATION
# =============================================================================
head("4. LAPIS 2 — KORELASI SPEARMAN TERHADAP PERGERAKAN SAHAM")

# Pemetaan tanggal terbit ke hari perdagangan
td = DF_STOCK[['dt']].rename(columns={'dt': 'trading_date'}).sort_values('trading_date')
pub = pd.DataFrame({'pub_date': DOCS['dt'].values, 'row': np.arange(N_VALID)}).sort_values('pub_date')

if WEEKEND_RULE == 'forward':
    m = pd.merge_asof(pub, td, left_on='pub_date', right_on='trading_date', direction='forward')
else:
    m = pub.merge(td, left_on='pub_date', right_on='trading_date', how='inner')

m = m.dropna(subset=['trading_date'])
n_luar_rentang = N_VALID - len(m)
row_idx = m['row'].values
day_key = m['trading_date'].values

print(f"Aturan hari non-perdagangan : '{WEEKEND_RULE}'")
print(f"Artikel terpetakan ke hari perdagangan : {len(m):,}"
      f"  (di luar rentang: {n_luar_rentang:,})")

# Agregasi harian dengan groupby atas matriks
agg_art = pd.DataFrame(BIN[row_idx].astype(np.int32), columns=VOCAB)
agg_tf = pd.DataFrame(CNT[row_idx], columns=VOCAB)
agg_art['_d'] = day_key
agg_tf['_d'] = day_key
D_ART = agg_art.groupby('_d').sum()
D_TF = agg_tf.groupby('_d').sum()
del agg_art, agg_tf

daily = (DF_STOCK.set_index('dt')[['return', 'abs_return']]
         .join(D_ART.add_prefix('art_'), how='left')
         .join(D_TF.add_prefix('tf_'), how='left')
         .fillna(0).sort_index())
daily['abs_return_t1'] = daily['abs_return'].shift(-1)
daily['return_t1'] = daily['return'].shift(-1)
DLAG = daily.dropna(subset=['abs_return_t1', 'return_t1'])
N_UJI = len(DLAG)

t_crit = stats.t.ppf(1 - ALPHA / 2, N_UJI - 2)
RHO_CRIT = t_crit / np.sqrt(N_UJI - 2 + t_crit ** 2)
print(f"Hari perdagangan untuk pengujian : {N_UJI:,}")
print(f"Nilai kritis |rho| pada alpha={ALPHA} : {RHO_CRIT:.4f}")

res, pvals_all = [], []
for kw in KEYWORDS_L1:
    c = KW_CLEAN[kw]
    xa, xt = DLAG[f'art_{c}'].values, DLAG[f'tf_{c}'].values
    r1, p1 = stats.spearmanr(xa, DLAG['abs_return'].values)
    r2, p2 = stats.spearmanr(xa, DLAG['abs_return_t1'].values)
    r3, p3 = stats.spearmanr(xa, DLAG['return_t1'].values)
    r4, p4 = stats.spearmanr(xt, DLAG['abs_return_t1'].values)
    quad = [(r1, p1), (r2, p2), (r3, p3), (r4, p4)]
    quad = [(0.0, 1.0) if np.isnan(r) else (r, p) for r, p in quad]
    (r1, p1), (r2, p2), (r3, p3), (r4, p4) = quad
    pvals_all += [p1, p2, p3, p4]
    n_sig = int(sum(p < ALPHA for p in (p1, p2, p3, p4)))
    res.append({'keyword': kw, 'kelompok': GROUP_MAP[kw],
                'n_hari_hadir': int((xa > 0).sum()),
                'rho_vol_d0': round(r1, 4), 'sig_vol_d0': '✓' if p1 < ALPHA else '✗',
                'rho_vol_t1': round(r2, 4), 'sig_vol_t1': '✓' if p2 < ALPHA else '✗',
                'rho_ret_t1': round(r3, 4), 'sig_ret_t1': '✓' if p3 < ALPHA else '✗',
                'rho_tf_t1': round(r4, 4), 'sig_tf_t1': '✓' if p4 < ALPHA else '✗',
                'n_signifikan': n_sig,
                'strength_score': round(np.mean(np.abs([r1, r2, r3, r4])), 6)})

df_l2 = (pd.DataFrame(res)
         .sort_values(['n_signifikan', 'strength_score'], ascending=[False, False])
         .reset_index(drop=True))
KEYWORDS_FINAL = df_l2.loc[df_l2['n_signifikan'] >= 1, 'keyword'].tolist()
N_FINAL = len(KEYWORDS_FINAL)

print(f"\nTABEL 4.3 — Hasil Spearman seluruh {len(KEYWORDS_L1)} kata kunci\n")
print(df_l2[['keyword', 'kelompok', 'rho_vol_d0', 'sig_vol_d0', 'rho_vol_t1', 'sig_vol_t1',
             'rho_ret_t1', 'sig_ret_t1', 'rho_tf_t1', 'sig_tf_t1', 'n_signifikan']]
      .to_string(index=False))
print(f"\nLolos Lapis 2 : {N_FINAL}   |   Gugur : {len(KEYWORDS_L1) - N_FINAL}")

# --- pengendalian kesalahan tipe I ---
pv = np.array(pvals_all)
M_UJI, N_SIG_TOTAL = pv.size, int((pv < ALPHA).sum())
EXP_SIG = M_UJI * ALPHA
bt = stats.binomtest(N_SIG_TOTAL, M_UJI, ALPHA, alternative='greater')
z_glob = (N_SIG_TOTAL - EXP_SIG) / np.sqrt(M_UJI * ALPHA * (1 - ALPHA))

print(f"\nUji binomial global : {N_SIG_TOTAL} dari {M_UJI} pengujian signifikan "
      f"(ekspektasi {EXP_SIG:.1f})")
print(f"                      z = {z_glob:.2f}   p = {bt.pvalue:.3e}")

kw_of_test = np.repeat(KEYWORDS_L1, 4)
fdr_rows = []
for q in FDR_LEVELS:
    mask = benjamini_hochberg(pv, q)
    kws = sorted(set(kw_of_test[mask]))
    fdr_rows.append({'Ambang FDR': f'q = {q:.2f}', 'Uji Lolos': int(mask.sum()),
                     'Kata Kunci': ', '.join(kws) if kws else '—'})
tabel44 = pd.DataFrame(fdr_rows)
print("\nTABEL 4.4 — Koreksi False Discovery Rate (Benjamini–Hochberg)\n")
print(tabel44.to_string(index=False))

_a = df_l2['rho_vol_t1'].values
_b = df_l2['rho_tf_t1'].values
print(f"\nKorelasi rho_vol_t1 vs rho_tf_t1 antarkata kunci : "
      f"{np.corrcoef(_a, _b)[0, 1]:.4f}")
print("   Mendekati 1 berarti kedua uji mengukur dimensi yang sama, sehingga")
print("   n_signifikan menghitung ganda satu bukti yang serupa.")

tabel45 = df_l2[df_l2['n_signifikan'] >= 1][['keyword', 'kelompok', 'n_signifikan']].copy()
tabel45['pengujian_signifikan'] = [
    ', '.join([n for n, s in
               zip(['ρ_vol_d0', 'ρ_vol_t1', 'ρ_ret_t1', 'ρ_tf_t1'],
                   [r['sig_vol_d0'], r['sig_vol_t1'], r['sig_ret_t1'], r['sig_tf_t1']])
               if s == '✓'])
    for _, r in df_l2[df_l2['n_signifikan'] >= 1].iterrows()]
tabel45.insert(0, 'No.', range(1, len(tabel45) + 1))
print(f"\nTABEL 4.5 — Daftar {N_FINAL} Kata Kunci Final\n")
print(tabel45.to_string(index=False))
print("\nSebaran per kelompok:")
for g in CANDIDATE_KEYWORDS:
    print(f"   {g:<26}: {int((tabel45['kelompok'] == g).sum())}")


# =============================================================================
# 5. LAPIS 3 — PENYARINGAN KORPUS
# =============================================================================
head("5. LAPIS 3 — PENYARINGAN KORPUS DENGAN KATA KUNCI FINAL")

j_final = [col_of[KW_CLEAN[k]] for k in KEYWORDS_FINAL]
mask_rel = BIN[:, j_final].any(axis=1)
N_RELEVAN = int(mask_rel.sum())

df_kontrib = (pd.DataFrame({'kata_kunci': KEYWORDS_FINAL,
                            'n_artikel': BIN[:, j_final].sum(axis=0)})
              .assign(pct_dari_relevan=lambda d: (d['n_artikel'] / max(N_RELEVAN, 1) * 100).round(2))
              .sort_values('n_artikel', ascending=False))
print(f"Artikel relevan : {N_RELEVAN:,}  ({N_RELEVAN / N_RAW * 100:.2f}% dari total scraping)")
print("\nKontribusi tiap kata kunci:\n")
print(df_kontrib.to_string(index=False))

rel_rows = set(np.nonzero(mask_rel)[0])
m_rel = m[m['row'].isin(rel_rows)]
N_HARI_ADA = int(pd.Series(m_rel['trading_date']).nunique())
N_HARI_KOSONG = N_TRADING - N_HARI_ADA
RATA_ARTIKEL = len(m_rel) / N_TRADING

df_daily_art = (pd.Series(m_rel['trading_date']).value_counts()
                .reindex(DF_STOCK['dt'], fill_value=0)
                .rename('n_artikel_relevan').rename_axis('trading_date').reset_index())

print(f"\nHari perdagangan total                   : {N_TRADING:,}")
print(f"Hari dengan minimal satu artikel relevan : {N_HARI_ADA:,}"
      f"  ({N_HARI_ADA / N_TRADING * 100:.2f}%)")
print(f"Hari tanpa artikel relevan               : {N_HARI_KOSONG:,}"
      f"  ({N_HARI_KOSONG / N_TRADING * 100:.2f}%)")
print(f"Rata-rata artikel relevan per hari       : {RATA_ARTIKEL:.2f}")

tabel46 = pd.DataFrame([
    ('Total artikel hasil scraping detikFinance', N_RAW, 100.0),
    ('Dokumen valid setelah penghapusan artikel kosong dan tidak terbaca',
     N_VALID, N_VALID / N_RAW * 100),
    (f'Artikel relevan setelah penyaringan {N_FINAL} kata kunci final',
     N_RELEVAN, N_RELEVAN / N_RAW * 100),
    ('Hari perdagangan dengan minimal satu artikel relevan',
     N_HARI_ADA, N_HARI_ADA / N_TRADING * 100),
], columns=['Tahapan', 'Jumlah', 'Persentase'])

head("TABEL 4.6 — RINGKASAN HASIL PENGUMPULAN DAN PENYARINGAN BERITA")
for _, r in tabel46.iterrows():
    print(f"{r['Tahapan']:<68} {_id(r['Jumlah']):>10} {_id(r['Persentase'], 2):>8}%")
print("-" * 96)
print("Persentase baris 1-3 terhadap total artikel scraping; "
      f"baris 4 terhadap {_id(N_TRADING)} hari perdagangan.")

print("\n--- Kalimat pengisi paragraf setelah Tabel 4.6 ---")
print(f"Dari {_id(N_TRADING)} hari perdagangan pada periode penelitian, sebanyak "
      f"{_id(N_HARI_KOSONG)} hari tidak memiliki artikel relevan dan diberi nilai nol "
      f"dengan penanda ketersediaan data. Rata-rata jumlah artikel relevan per hari "
      f"perdagangan adalah {_id(RATA_ARTIKEL, 2)} artikel.")


# =============================================================================
# 6. EKSPOR
# =============================================================================
head("6. EKSPOR")

cols_keep = [c for c in [DATE_COL, TITLE_COL, TEXT_COL, URL_COL] if c]
df_relevan = DF_NEWS_RAW.loc[IDX_OK[mask_rel], cols_keep].copy()
df_relevan['date'] = DOCS.loc[mask_rel, 'dt'].dt.date.values
df_relevan['text_clean'] = DOCS.loc[mask_rel, 'text'].values

df_relevan.to_csv('korpus_relevan_final.csv', index=False)
df_daily_art.to_csv('artikel_relevan_harian.csv', index=False)
daily.reset_index().to_csv('frekuensi_keyword_harian.csv', index=False)

with pd.ExcelWriter('validasi_keyword_lengkap.xlsx', engine='openpyxl') as w:
    tabel41.to_excel(w, sheet_name='Tabel_4_1_Kandidat')
    df_l1.to_excel(w, sheet_name='Tabel_4_2_Lapis1', index=False)
    df_l2.to_excel(w, sheet_name='Tabel_4_3_Lapis2', index=False)
    tabel44.to_excel(w, sheet_name='Tabel_4_4_FDR', index=False)
    tabel45.to_excel(w, sheet_name='Tabel_4_5_Final', index=False)
    tabel46.to_excel(w, sheet_name='Tabel_4_6_Funnel', index=False)
    df_kontrib.to_excel(w, sheet_name='Kontribusi_Keyword', index=False)

print("  korpus_relevan_final.csv        → masukan skoring VADER dan FinBERT")
print("  artikel_relevan_harian.csv      → jumlah artikel per hari perdagangan")
print("  frekuensi_keyword_harian.csv    → frekuensi harian tiap kata kunci")
print("  validasi_keyword_lengkap.xlsx   → seluruh Tabel 4.1 sampai 4.6")

head("RINGKASAN ANGKA UNTUK BAB 4")
print(f"  Kandidat kata kunci                : {len(ALL_CANDIDATES)}")
print(f"  Lolos Lapis 1 (TF-IDF)             : {len(KEYWORDS_L1)}")
print(f"  Lolos Lapis 2 (Spearman)           : {N_FINAL}")
print(f"  Total artikel scraping             : {_id(N_RAW)}")
print(f"  Dokumen valid                      : {_id(N_VALID)}")
print(f"  Artikel relevan                    : {_id(N_RELEVAN)}")
print(f"  Hari perdagangan                   : {_id(N_TRADING)}")
print(f"  Hari dengan artikel relevan        : {_id(N_HARI_ADA)}")
print(f"  Hari perdagangan diuji Lapis 2     : {_id(N_UJI)}")
print(f"  Nilai kritis |rho|                 : {RHO_CRIT:.4f}")
print(f"  Uji signifikan / total             : {N_SIG_TOTAL} / {M_UJI}  (z = {z_glob:.2f})")
print(f"\n  df_relevan siap dipakai tahap sentimen : {len(df_relevan):,} artikel")


TABEL 4.1 — DISTRIBUSI KANDIDAT KATA KUNCI PER KELOMPOK TEMATIK
         Kelompok Tematik  Jumlah Kandidat
1       Energi Terbarukan               21
2            Energi Fosil               15
3  Kebijakan & Lingkungan               11
4             Kelistrikan                7
5                  Nuklir                4
Total                           58

1. PEMUATAN DATA
  2016 :  22,539 artikel
  2017 :  25,536 artikel
  2018 :  26,574 artikel
  2019 :  26,938 artikel
  2020 :  34,438 artikel
  2021 :  34,764 artikel
  2022 :  34,348 artikel
  2023 :  34,548 artikel
  2024 :  33,011 artikel
  2025 :  30,561 artikel
Total artikel mentah : 303,257
Kolom  → tanggal: 'tanggal'  teks: 'isi'  judul: 'judul'  url: 'None'

Artikel teks kosong / terlalu pendek : 579
Artikel tanggal tidak terbaca        : 0
Dokumen valid                        : 302,678
Duplikat isi (diagnostik, tidak dibuang) : 1,458
Duplikat URL (diagnostik, tidak dibuang) : kolom URL tidak ada
Rentang tanggal : 2016-01-01 